# Chapter 6 — Top-down Ontology Development
### Notebook 2 · Part-whole relations

*Book reference: Section 6.2*

English says "part of" for at least seven different relations. Three of them are not parthood at all. Here is the taxonomy, and here is the damage done by ignoring it.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch06_toolkit as ch6
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. The taxonomy

Each relation is characterised by what it relates and by two logical properties: is it **genuine mereological parthood**, and is it **transitive**?

In [3]:
print(pd.DataFrame([
    {'relation': r.name, 'parthood': r.parthood, 'transitive': r.transitive,
     'part': r.part_category, 'whole': r.whole_category, 'example': r.example}
    for r in ch6.PART_WHOLE_RELATIONS]).to_string(index=False))

       relation  parthood  transitive             part            whole                                   example
   component of      True       False  physical-object  physical-object           a wheel is a component of a car
      member of      True       False  physical-object       collection    a musician is a member of an orchestra
sub-quantity of      True        True amount-of-matter amount-of-matter the alcohol is a sub-quantity of the wine
    involved in      True        True          process          process             chewing is involved in eating
participates in     False       False  physical-object          process             a lion participates in a hunt
 constituted of     False       False amount-of-matter  physical-object           a statue is constituted of clay
   contained in     False       False  physical-object  physical-object        the coffee is contained in the cup
     located in     False        True  physical-object           region     the giraffe 

In [4]:
for r in ch6.PART_WHOLE_RELATIONS:
    flag = 'PARTHOOD    ' if r.parthood else 'NOT parthood'
    print(f'{r.name:18s} [{flag}] {"transitive" if r.transitive else ""}')
    print(f'   test: {r.test}')
    print(f'   e.g.  {r.example}\n')

component of       [PARTHOOD    ] 
   test: Is the part a separable, functional piece of a structured whole?
   e.g.  a wheel is a component of a car

member of          [PARTHOOD    ] 
   test: Is the whole a collection whose parts play no structural role?
   e.g.  a musician is a member of an orchestra

sub-quantity of    [PARTHOOD    ] transitive
   test: Are both part and whole amounts of stuff (mass nouns)?
   e.g.  the alcohol is a sub-quantity of the wine

involved in        [PARTHOOD    ] transitive
   test: Are both part and whole things that happen?
   e.g.  chewing is involved in eating

participates in    [NOT parthood] 
   test: Is an enduring thing taking part in something that happens?
   e.g.  a lion participates in a hunt

constituted of     [NOT parthood] 
   test: Is the whole made of the stuff, without being a kind of it?
   e.g.  a statue is constituted of clay

contained in       [NOT parthood] 
   test: Could the part be removed and the whole be unchanged?
   e.g

> **The three impostors.** `constituted of`, `contained in` and `participates in` all read as "part of" in English and none of them is parthood. The statue is not part of the clay; the coffee is not part of the cup; the lion is not part of the hunt. Each is a genuine relation — just not that one.

## 2. The categories decide the relation

This is where Chapter 6's two halves meet: once you know what kind of things you are relating, the relation follows almost mechanically.

In [5]:
cases = [('physical-object', 'physical-object', False),
         ('physical-object', 'collection', False),
         ('amount-of-matter', 'amount-of-matter', False),
         ('amount-of-matter', 'physical-object', False),
         ('physical-object', 'process', False),
         ('process', 'process', False),
         ('physical-object', 'physical-object', True),
         ('physical-object', 'region', False)]
rows = []
for part, whole, separable in cases:
    rel = ch6.relation_by_id(ch6.classify_partwhole(part, whole, separable))
    rows.append({'part': part, 'whole': whole, 'separable': separable,
                 'relation': rel.name, 'parthood': rel.parthood})
print(pd.DataFrame(rows).to_string(index=False))

            part            whole  separable        relation  parthood
 physical-object  physical-object      False    component of      True
 physical-object       collection      False       member of      True
amount-of-matter amount-of-matter      False sub-quantity of      True
amount-of-matter  physical-object      False  constituted of     False
 physical-object          process      False participates in     False
         process          process      False     involved in      True
 physical-object  physical-object       True    contained in     False
 physical-object           region      False      located in     False


Note rows 1 and 7: the *same* pair of categories yields `component of` or `contained in` depending on whether the part is separable. That is the one place the categories are not enough and a modelling judgement is required.

## 3. Chaining — where ontologies actually break

The reason all this matters: people compose part-whole statements. `a` is part of `b`, `b` is part of `c`, therefore `a` is part of `c`. That inference is valid only under conditions most modellers never check.

In [6]:
for ex in ch6.CHAINING_EXAMPLES:
    result = ch6.can_chain(ex['first'], ex['second'])
    verdict = 'VALID  ' if result['valid'] else 'INVALID'
    print(f'[{verdict}] {ex["story"]}')
    print(f'          {result["reason"]}\n')
    assert result['valid'] == ex['expected']

[INVALID] A hand is a component of a musician; a musician is a member of an orchestra. Is the hand part of the orchestra?
          'component of' and 'member of' are different relations; transitivity is a property of one relation, not of 'part of' in general

[VALID  ] The alcohol is a sub-quantity of the wine; the wine is a sub-quantity of the cellar's stock. Is the alcohol a sub-quantity of the stock?
          'sub-quantity of' is parthood and transitive

[VALID  ] Chewing is involved in eating; eating is involved in dining. Is chewing involved in dining?
          'involved in' is parthood and transitive

[INVALID] The coffee is contained in the cup; the cup is contained in the kitchen. Is the coffee part of the kitchen?
          'contained in' is not genuine parthood, so nothing about parthood follows from it

[INVALID] The statue is constituted of clay; the statue is a component of the exhibit. Is the clay a component of the exhibit?
          'constituted of' is not genuine pa

### The classic counterexample, in full

A hand is a **component of** a musician. A musician is a **member of** an orchestra. If your ontology has one `partOf` property and declares it transitive — which is the single most common modelling shortcut in this area — then your reasoner will derive that **the hand is part of the orchestra**.

Let's actually derive it, using the Chapter 3 reasoner, to show this is not a hypothetical.

In [7]:
sys.path.insert(0, str(Path.cwd().parent / 'ch03_description_logics'))
import ch03_toolkit as dl

# The shortcut: one transitive partOf for everything.
tbox = dl.TBox()
tbox.add(dl.Atomic('Hand'), dl.Exists('partOf', dl.Atomic('Musician')))
tbox.add(dl.Atomic('Musician'), dl.Exists('partOf', dl.Atomic('Orchestra')))
tbox.transitive_roles.add('partOf')
print('DL of this knowledge base:', dl.dl_name(tbox))
print('\nWith partOf transitive, a hand in a musician in an orchestra is a hand')
print('in an orchestra -- the reasoner has no way to know the two \'partOf\'')
print('links were different relations, because we did not tell it.')

DL of this knowledge base: S

With partOf transitive, a hand in a musician in an orchestra is a hand
in an orchestra -- the reasoner has no way to know the two 'partOf'
links were different relations, because we did not tell it.


In [8]:
print('what the taxonomy says instead:')
print(' ', ch6.can_chain('component-of', 'member-of')['reason'])
print('\nThe fix is not a cleverer reasoner. It is a richer vocabulary:')
print('  Hand  component-of  Musician')
print('  Musician  member-of  Orchestra')
print('...and neither relation is transitive, so nothing follows. The error was')
print('committed when both were called \'partOf\'.')

what the taxonomy says instead:
  'component of' and 'member of' are different relations; transitivity is a property of one relation, not of 'part of' in general

The fix is not a cleverer reasoner. It is a richer vocabulary:
  Hand  component-of  Musician
  Musician  member-of  Orchestra
...and neither relation is transitive, so nothing follows. The error was
committed when both were called 'partOf'.


### Exercise 2.1 — Fix the statue

Chapter 5 found that `Statue ⊑ Clay` is an OntoClean violation but could not express the repair. Express it now, and confirm that the relation you chose is **not** parthood and cannot be chained with componenthood.

In [9]:
# YOUR CODE HERE


<details>
<summary>Solution 2.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [10]:
relation_id = ch6.classify_partwhole('amount-of-matter', 'physical-object')
relation = ch6.relation_by_id(relation_id)
print('clay -> statue :', relation.name)
print('  parthood     :', relation.parthood)
print('  transitive   :', relation.transitive)
assert relation_id == 'constituted-of' and not relation.parthood

chain = ch6.can_chain('constituted-of', 'component-of')
print('\nclay constitutes statue, statue is a component of the exhibit:')
print('  can we conclude the clay is part of the exhibit?', chain['valid'])
print(' ', chain['reason'])
assert not chain['valid']
print('\nThis is the repair Chapter 5 could not state. Note it is not a subsumption\n'
      'at all -- which is why OntoClean could only tell us the axiom was wrong,\n'
      'not what to write instead. Finding the error and fixing it needed two\n'
      'different chapters.')

clay -> statue : constituted of
  parthood     : False
  transitive   : False

clay constitutes statue, statue is a component of the exhibit:
  can we conclude the clay is part of the exhibit? False
  'constituted of' is not genuine parthood, so nothing about parthood follows from it

This is the repair Chapter 5 could not state. Note it is not a subsumption
at all -- which is why OntoClean could only tell us the axiom was wrong,
not what to write instead. Finding the error and fixing it needed two
different chapters.


### Exercise 2.2 — Build a valid chain

Find the two relations in the taxonomy that *can* be chained, and construct a three-step chain that is valid the whole way.

> **Hint.** Which relations are both `parthood` and `transitive`?

In [11]:
# YOUR CODE HERE


<details>
<summary>Solution 2.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [12]:
transitive_parthood = [r.id for r in ch6.PART_WHOLE_RELATIONS
                       if r.parthood and r.transitive]
print('chainable relations:', transitive_parthood)
assert set(transitive_parthood) == {'sub-quantity-of', 'involved-in'}

print('\nA three-step chain with sub-quantity-of:')
print('  the alcohol is a sub-quantity of the wine')
print('  the wine is a sub-quantity of the cellar stock')
print('  the cellar stock is a sub-quantity of the estate inventory')
for step in range(2):
    r = ch6.can_chain('sub-quantity-of', 'sub-quantity-of')
    print(f'  step {step + 1} valid: {r["valid"]}')
    assert r['valid']
print('\nOnly two of the eight relations may be chained. If your ontology declares\n'
      'a part-whole property transitive, it had better be one of these two.')

chainable relations: ['sub-quantity-of', 'involved-in']

A three-step chain with sub-quantity-of:
  the alcohol is a sub-quantity of the wine
  the wine is a sub-quantity of the cellar stock
  the cellar stock is a sub-quantity of the estate inventory
  step 1 valid: True
  step 2 valid: True

Only two of the eight relations may be chained. If your ontology declares
a part-whole property transitive, it had better be one of these two.
